# 投资时钟策略复现

本Notebook复现华泰证券《行业配置策略：投资时钟视角》研报的核心策略

**数据源**: tushare > akshare > baostock > efinance > yfinance
**数据说明**: 无法获取的真实数据使用模拟数据替代

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print('环境配置完成')

## 1. 数据获取

In [ ]:
from source.data_fetcher import DataFetcher

fetcher = DataFetcher()
print('数据获取器初始化完成')
print(f"tushare可用: {fetcher.tushare_available}")
print(f"akshare可用: {fetcher.akshare_available}")
print(f"baostock可用: {fetcher.baostock_available}")
print(f"efinance可用: {fetcher.efinance_available}")
print(f"yfinance可用: {fetcher.yfinance_available}")

### 1.1 获取指数数据（真实数据）

In [ ]:
# 获取沪深300指数数据
hs300_df = fetcher.get_index_daily('000300.SH', start_date='20100101', end_date='20210630')
print(f'沪深300数据: {len(hs300_df)} 条记录')
if not hs300_df.empty:
    print(hs300_df.head())

In [ ]:
# 获取更多指数数据
zz500_df = fetcher.get_index_daily('000905.SH', start_date='20100101', end_date='20210630')
cybz_df = fetcher.get_index_daily('399006.SZ', start_date='20100101', end_date='20210630')

print(f'中证500数据: {len(zz500_df)} 条记录')
print(f'创业板指数据: {len(cybz_df)} 条记录')

In [ ]:
# 获取国债指数数据
bond_df = fetcher.get_bond_yield(start_date='20100101', end_date='20210630')
print(f'国债收益率数据: {len(bond_df)} 条记录')
if not bond_df.empty:
    print(bond_df.head())

### 1.2 获取大类资产收益率

In [ ]:
# 获取大类资产收益率数据
asset_returns = fetcher.get_asset_returns(start_date='20110101', end_date='20210630')
print(f'资产收益率数据形状: {asset_returns.shape}')
print(asset_returns.describe())

### 1.3 获取行业收益率

In [ ]:
# 获取行业收益率数据
industry_returns = fetcher.get_industry_returns(start_date='20110101', end_date='20210630')
print(f'行业收益率数据形状: {industry_returns.shape}')
print(f'行业列表: {list(industry_returns.columns)}')

## 2. 因子合成

In [ ]:
from source.factor_synthesis import FactorSynthesis

synthesizer = FactorSynthesis()
print('因子合成器初始化完成')

In [ ]:
# 获取宏观因子（模拟42个月周期）
factor_dict = fetcher.get_all_macro_factors(start_date='20100101', end_date='20210630')

print('宏观因子数据:')
for name, series in factor_dict.items():
    print(f"  {name}: {len(series)} 期数据, 均值={series.mean():.2f}, 标准差={series.std():.2f}")

## 3. 因子预测

In [ ]:
from source.factor_predictor import FactorPredictor

predictor = FactorPredictor(cycle_period=42)
print('因子预测器初始化完成 (42个月周期)')

In [ ]:
# 获取各因子的预测观点
factor_predictions = predictor.get_all_predictions(factor_dict)

print('各因子观点统计:')
for factor_name, predictions in factor_predictions.items():
    print(f"\n{factor_name}:")
    for method, views in predictions.items():
        if len(views) > 0:
            up_count = (views == 1).sum()
            down_count = (views == -1).sum()
            neutral_count = (views == 0).sum()
            print(f"  {method}: 上行={up_count}, 下行={down_count}, 中性={neutral_count}")

## 4. 宏观-资产映射

In [ ]:
from source.asset_mapping import AssetMapping

asset_mapper = AssetMapping()
print('资产映射器初始化完成')

In [ ]:
# 构建宏观-资产映射关系
asset_mapping = asset_mapper.build_macro_asset_mapping(factor_dict, asset_returns)

print('宏观-资产映射结果:')
for factor_name, mapping in asset_mapping.items():
    print(f"\n{factor_name}:")
    for asset_name, regime_data in list(mapping.items())[:3]:
        print(f"  {asset_name}: {regime_data}")

## 5. 投资时钟构建

In [ ]:
# 构建增长-通胀投资时钟
clock_result = asset_mapper.build_growth_inflation_clock(
    factor_dict['growth'],
    factor_dict['inflation'],
    asset_returns
)

print('增长-通胀投资时钟状态分布:')
if 'clock' in clock_result:
    clock_series = pd.Series(clock_result['clock'])
    print(clock_series.value_counts())

In [ ]:
# 构建信用-货币投资时钟
credit_clock = asset_mapper.build_credit_monetary_clock(
    factor_dict['credit'],
    factor_dict['monetary'],
    asset_returns
)

print('信用-货币投资时钟状态分布:')
if 'clock' in credit_clock:
    clock_series = pd.Series(credit_clock['clock'])
    print(clock_series.value_counts())

## 6. 大类资产配置策略回测

In [ ]:
from source.asset_strategy import AssetStrategy
from source.backtest import BacktestEngine

asset_strategy = AssetStrategy(
    target_volatility=0.05,
    max_leverage=2.0,
    risk_free_rate=0.04
)

backtester = BacktestEngine(
    initial_capital=1000000,
    transaction_cost=0.002,
    risk_free_rate=0.04
)

print('策略和回测引擎初始化完成')

In [ ]:
# 运行大类资产配置回测
asset_results = backtester.run_asset_backtest(
    asset_strategy,
    asset_returns,
    factor_dict,
    asset_mapping,
    start_date='2011-01-01',
    end_date='2021-06-30'
)

print('大类资产配置回测完成!')

## 7. 行业轮动策略回测

In [ ]:
from source.industry_strategy import IndustryStrategy

industry_strategy = IndustryStrategy(top_n=5, momentum_window=20)
print('行业轮动策略初始化完成')

In [ ]:
# 构建行业映射
industry_mapping = asset_mapper.build_industry_mapping(factor_dict, industry_returns)

print(f'行业映射构建完成，共 {len(industry_mapping)} 个因子')

In [ ]:
# 运行行业轮动回测
industry_results = backtester.run_industry_backtest(
    industry_strategy,
    industry_returns,
    factor_dict,
    industry_mapping,
    benchmark_returns=asset_returns['沪深300'] if '沪深300' in asset_returns.columns else None,
    start_date='2011-01-01',
    end_date='2021-06-30'
)

print('行业轮动回测完成!')

## 8. 结果对比

In [ ]:
# 策略对比表格
comparison = backtester.compare_strategies(
    [asset_results, industry_results],
    ['大类资产策略', '行业轮动策略']
)

print('='*70)
print('策略绩效对比')
print('='*70)
print(comparison.to_string(index=False))

## 9. 可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 大类资产策略 - 组合价值
if 'portfolio_values' in asset_results and len(asset_results['portfolio_values']) > 0:
    values = asset_results['portfolio_values']
    axes[0, 0].plot(values.index, values.values, 'b-', linewidth=2)
    axes[0, 0].set_title('大类资产策略 - 组合价值', fontsize=14)
    axes[0, 0].set_xlabel('日期')
    axes[0, 0].set_ylabel('组合价值')
    axes[0, 0].grid(True, alpha=0.3)

# 行业轮动策略 - 组合价值
if 'portfolio_values' in industry_results and len(industry_results['portfolio_values']) > 0:
    values = industry_results['portfolio_values']
    axes[0, 1].plot(values.index, values.values, 'r-', linewidth=2)
    axes[0, 1].set_title('行业轮动策略 - 组合价值', fontsize=14)
    axes[0, 1].set_xlabel('日期')
    axes[0, 1].set_ylabel('组合价值')
    axes[0, 1].grid(True, alpha=0.3)

# 宏观因子走势
for factor_name, series in factor_dict.items():
    if len(series) > 0:
        axes[1, 0].plot(series.index, series.values, label=factor_name, linewidth=1.5)
axes[1, 0].set_title('宏观因子走势', fontsize=14)
axes[1, 0].set_xlabel('日期')
axes[1, 0].set_ylabel('因子值')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 累计收益对比
if 'returns_series' in asset_results and 'returns_series' in industry_results:
    asset_cum = (1 + asset_results['returns_series']).cumprod()
    industry_cum = (1 + industry_results['returns_series']).cumprod()
    
    axes[1, 1].plot(asset_cum.index, asset_cum.values, label='大类资产', linewidth=2)
    axes[1, 1].plot(industry_cum.index, industry_cum.values, label='行业轮动', linewidth=2)
    axes[1, 1].set_title('累计收益对比', fontsize=14)
    axes[1, 1].set_xlabel('日期')
    axes[1, 1].set_ylabel('累计收益')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/results/investment_clock_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存至 output/results/investment_clock_results.png')

## 10. 详细绩效指标

In [ ]:
def print_strategy_summary(name, results):
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"累计收益:   {results.get('cumulative_return', 0):.2%}")
    print(f"年化收益:   {results.get('annual_return', 0):.2%}")
    print(f"年化波动:   {results.get('annual_volatility', 0):.2%}")
    print(f"夏普比率:   {results.get('sharpe_ratio', 0):.2f}")
    print(f"最大回撤:   {results.get('max_drawdown', 0):.2%}")
    print(f"卡玛比率:   {results.get('calmar_ratio', 0):.2f}")
    print(f"月度胜率:   {results.get('win_rate', 0):.2%}")
    if 'tracking_error' in results:
        print(f"跟踪误差:   {results.get('tracking_error', 0):.2%}")
    if 'information_ratio' in results:
        print(f"信息比率:   {results.get('information_ratio', 0):.2f}")

print_strategy_summary('大类资产投资时钟策略', asset_results)
print_strategy_summary('行业轮动投资时钟策略', industry_results)

## 11. 因子观点分析

In [ ]:
# 分析最近12个月的因子观点
print('最近12个月因子观点:')
for factor_name, predictions in factor_predictions.items():
    if 'combined' in predictions:
        combined = predictions['combined']
        recent = combined.tail(12)
        print(f"\n{factor_name}:")
        for date, view in recent.items():
            view_str = '上行' if view == 1 else ('下行' if view == -1 else '中性')
            print(f"  {date.strftime('%Y-%m')}: {view_str}")

## 12. 结论

本notebook完成了以下工作：

1. **数据获取**: 使用tushare/akshare/baostock/efinance/yfinance获取真实市场数据

2. **因子合成**: 基于42个月周期生成增长、通胀、信用、货币四个宏观因子

3. **因子预测**: 实现相位判断法和因子动量法预测因子方向

4. **投资时钟**: 构建增长-通胀、信用-货币双轮驱动投资时钟

5. **回测验证**: 
   - 大类资产策略
   - 行业轮动策略

**注意**: 由于宏观数据获取限制，本示例使用了模拟宏观因子数据。实际应用中应使用真实的宏观指标数据。